# Order block and range context outputs

This deterministic CPU example illustrates a software contract only. It uses no market feed and provides no evidence of trading performance.

In [1]:
import numpy as np
import pandas as pd

from features.context.order_blocks import build_order_block_features
from features.context.premium_discount import build_pd_context_features
from features.technical.indicators import atr

idx = pd.date_range("2026-01-05 00:00", periods=48, freq="1h", tz="Etc/GMT-3")
close = pd.Series(np.linspace(100.0, 104.7, len(idx)) + np.sin(np.arange(len(idx)) / 2), index=idx)
bars = pd.DataFrame(
    {"open": close - 0.2, "high": close + 0.4, "low": close - 0.5, "close": close}, index=idx
)
blocks = build_order_block_features(
    bars, swing_period=2, impulse_atr=0.5, search_bars=4, max_age_bars=12, atr_period=3
)
pd_features = build_pd_context_features(bars, atr(bars, period=3))
assert blocks.index.equals(bars.index)
assert {"ob_bull_active", "ob_bear_active", "ob_bull_low", "ob_bear_high"}.issubset(blocks.columns)
assert pd_features.index.equals(bars.index)
assert {"ctx_in_premium", "ctx_in_discount", "htf_day_bias_available"}.issubset(pd_features.columns)
assert ((pd_features["ctx_in_premium"] + pd_features["ctx_in_discount"]) <= 1).all()
(
    blocks[["ob_bull_active", "ob_bear_active"]].tail(),
    pd_features[["ctx_in_premium", "ctx_in_discount"]].tail(),
)

(                           ob_bull_active  ob_bear_active
 2026-01-06 19:00:00+03:00             0.0             0.0
 2026-01-06 20:00:00+03:00             0.0             0.0
 2026-01-06 21:00:00+03:00             0.0             0.0
 2026-01-06 22:00:00+03:00             0.0             0.0
 2026-01-06 23:00:00+03:00             0.0             0.0,
                            ctx_in_premium  ctx_in_discount
 2026-01-06 19:00:00+03:00             1.0              0.0
 2026-01-06 20:00:00+03:00             1.0              0.0
 2026-01-06 21:00:00+03:00             1.0              0.0
 2026-01-06 22:00:00+03:00             1.0              0.0
 2026-01-06 23:00:00+03:00             1.0              0.0)